In [1]:
from dotenv import load_dotenv
import os, sys, json, re, subprocess, tempfile

load_dotenv()

from astrapy import DataAPIClient
from github import Auth, Github
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

# --- RAG setup ---
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
client = DataAPIClient(os.getenv("ASTRA_DB_APPLICATION_TOKEN"))
db = client.get_database(os.getenv("ASTRA_DB_API_ENDPOINT"))
collection = db.get_collection("codeguardian_style_corpus")

def retrieve_style_context(diff_text: str, top_k: int = 5) -> str:
    query_vector = embeddings.embed_query(diff_text)
    results = collection.find(sort={"$vector": query_vector}, limit=top_k)
    return "\n\n".join(f"[{r['type']} — {r['source']}]\n{r['text']}" for r in results)

# --- GitHub + LLM setup ---
gh = Github(auth=Auth.Token(os.getenv("GITHUB_PAT")))
repo = gh.get_repo("PrashantAghara/fastapi")
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

current_pr = None  # set at call time by each node

# --- Shared diff helpers ---
def get_changed_line_ranges(patch: str) -> list[tuple[int, int]]:
    ranges = []
    for match in re.finditer(r"@@ -\d+,?\d* \+(\d+),?(\d*) @@", patch):
        start = int(match.group(1))
        length = int(match.group(2)) if match.group(2) else 1
        ranges.append((start, start + length - 1))
    return ranges

def get_pr_diff_text(pr, filenames: list[str]) -> str:
    diff_parts = []
    for f in pr.get_files():
        if f.filename in filenames:
            diff_parts.append(f"--- {f.filename} ---\n{f.patch}")
    return "\n\n".join(diff_parts)

def get_full_file_content(pr, filename: str) -> str:
    return repo.get_contents(filename, ref=pr.head.sha).decoded_content.decode("utf-8")

# --- Static Analysis (ruff) ---
def run_ruff_on_file(filename: str, full_content: str) -> list[dict]:
    with tempfile.NamedTemporaryFile(suffix=".py", delete=False, mode="w", encoding="utf-8") as tmp:
        tmp.write(full_content)
        tmp_path = tmp.name
    result = subprocess.run(["ruff", "check", tmp_path, "--output-format=json"], capture_output=True, text=True)
    os.unlink(tmp_path)
    try:
        return json.loads(result.stdout) if result.stdout else []
    except json.JSONDecodeError:
        return []

def filter_to_diff(findings: list[dict], ranges: list[tuple[int, int]], row_key=lambda f: f["location"]["row"]) -> list[dict]:
    return [f for f in findings if any(s <= row_key(f) <= e for s, e in ranges)]

@tool
def static_analysis_tool(filename: str) -> dict:
    """Run ruff against this PR's version of a file, scoped to only the changed lines."""
    file_obj = next(f for f in current_pr.get_files() if f.filename == filename)
    full_content = get_full_file_content(current_pr, filename)
    raw = run_ruff_on_file(filename, full_content)
    ranges = get_changed_line_ranges(file_obj.patch)
    return {"filename": filename, "findings": filter_to_diff(raw, ranges)}

static_analysis_agent = create_agent(
    model=llm,
    tools=[static_analysis_tool],
    system_prompt=(
        "You are a Static Analysis Agent reviewing a pull request. "
        "Call static_analysis_tool once per given filename. "
        "When summarizing, use the EXACT 'location.row' value as the Line number — never estimate. "
        "Summarize findings grouped by severity, end with a one-line verdict: PASS, PASS_WITH_WARNINGS, or FAIL."
    ),
)

# --- Style Agent (two-pass hybrid) ---
BASELINE_RULES = """
- All function signatures must have type hints on parameters and return values
- Public functions must have a docstring describing purpose, args, and return value
- Function and variable names must be descriptive, not abbreviated (e.g. `user_id` not `uid`)
- Avoid nesting conditionals more than 3 levels deep — prefer early returns
"""

baseline_prompt = """You are reviewing a pull request diff against ONLY these baseline rules:
{baseline_rules}

Diff:
{diff}

For each violation: filename, line (from diff hunk header), comment, severity (info/warning).
If none, say so explicitly."""

project_context_prompt = """You are reviewing a pull request diff against ONLY the conventions evident in this project's actual codebase (retrieved below) — not general best practices.

Retrieved project context:
{retrieved_context}

Diff:
{diff}

Identify anything in the diff that deviates from patterns clearly shown in the retrieved context above.
For each: filename, line (from diff hunk header), comment, severity (info/warning).
If the context doesn't clearly support a finding, say so rather than guessing."""

def run_style_agent_two_pass(pr, filenames: list[str]) -> str:
    diff_text = get_pr_diff_text(pr, filenames)
    retrieved_context = retrieve_style_context(diff_text)

    baseline_result = llm.invoke(baseline_prompt.format(baseline_rules=BASELINE_RULES, diff=diff_text))
    context_result = llm.invoke(project_context_prompt.format(retrieved_context=retrieved_context, diff=diff_text))

    merge_prompt = f"""Combine these two independent review passes into one final report. Keep each finding's source labeled.

BASELINE PASS RESULTS:
{baseline_result.content}

PROJECT-CONTEXT PASS RESULTS:
{context_result.content}

Output a combined list of findings (deduplicated if any overlap), each labeled [baseline] or [project-context], then end with one overall verdict: PASS, PASS_WITH_WARNINGS, or FAIL."""

    return llm.invoke(merge_prompt).content

# --- Security Agent (bandit) ---
def run_bandit_on_file(filename: str, full_content: str) -> list[dict]:
    with tempfile.NamedTemporaryFile(suffix=".py", delete=False, mode="w", encoding="utf-8") as tmp:
        tmp.write(full_content)
        tmp_path = tmp.name
    result = subprocess.run(["bandit", "-f", "json", tmp_path], capture_output=True, text=True)
    os.unlink(tmp_path)
    try:
        data = json.loads(result.stdout)
        return data.get("results", [])
    except json.JSONDecodeError:
        return []

def filter_bandit_to_diff(findings: list[dict], ranges: list[tuple[int, int]]) -> list[dict]:
    return [f for f in findings if any(s <= f["line_number"] <= e for s, e in ranges)]

@tool
def security_analysis_tool(filename: str) -> dict:
    """Run bandit against this PR's version of a file, scoped to only the changed lines, to find security issues."""
    file_obj = next(f for f in current_pr.get_files() if f.filename == filename)
    full_content = get_full_file_content(current_pr, filename)
    raw = run_bandit_on_file(filename, full_content)
    ranges = get_changed_line_ranges(file_obj.patch)
    filtered = filter_bandit_to_diff(raw, ranges)
    return {
        "filename": filename,
        "findings": [
            {"line": f["line_number"], "issue": f["issue_text"], "severity": f["issue_severity"],
             "confidence": f["issue_confidence"], "test_id": f["test_id"]}
            for f in filtered
        ],
    }

security_agent = create_agent(
    model=llm,
    tools=[security_analysis_tool],
    system_prompt=(
        "You are a Security Agent reviewing a pull request for security risks. "
        "Call security_analysis_tool once per given filename. "
        "Use the EXACT 'line' value from each finding — never estimate. "
        "Summarize findings grouped by severity, and end with a one-line verdict: PASS, PASS_WITH_WARNINGS, or FAIL. "
        "Any 'high' severity finding should always result in FAIL, regardless of how few findings there are."
    ),
)

print("Setup complete — all Phase 1-3 agents ready")

d:\resume-projects\codegaurdian\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10420.76it/s]


Setup complete — all Phase 1-3 agents ready


In [2]:
from typing import TypedDict, Optional

class ReviewState(TypedDict):
    pr: object
    pr_title: str
    pr_body: str
    py_filenames: list[str]
    static_result: Optional[str]
    style_result: Optional[str]
    security_result: Optional[str]
    final_summary: Optional[str]
    skip_reason: Optional[str]

In [3]:
def gather_context_node(state: ReviewState) -> ReviewState:
    pr = state["pr"]
    py_filenames = [f.filename for f in pr.get_files() if f.filename.endswith(".py")]
    return {
        **state,
        "pr_title": pr.title,
        "pr_body": pr.body or "(no description provided)",
        "py_filenames": py_filenames,
    }

In [4]:
def supervisor_node(state: ReviewState) -> ReviewState:
    if not state["py_filenames"]:
        return {**state, "skip_reason": "No Python files changed — nothing for CodeGuardian to review."}
    return state

def route_after_supervisor(state: ReviewState) -> str:
    return "skip" if state.get("skip_reason") else "review"

In [5]:
def static_analysis_node(state: ReviewState) -> ReviewState:
    global current_pr
    current_pr = state["pr"]
    result = static_analysis_agent.invoke({
        "messages": [{"role": "user", "content": f"Changed Python files: {state['py_filenames']}"}]
    })["messages"][-1].content
    return {**state, "static_result": result}

def style_node(state: ReviewState) -> ReviewState:
    result = run_style_agent_two_pass(state["pr"], state["py_filenames"])
    return {**state, "style_result": result}

def security_node(state: ReviewState) -> ReviewState:
    global current_pr
    current_pr = state["pr"]
    result = security_agent.invoke({
        "messages": [{"role": "user", "content": f"Changed Python files: {state['py_filenames']}"}]
    })["messages"][-1].content
    return {**state, "security_result": result}

In [6]:
summarizer_prompt_v2 = """You are a Summarizer Agent producing a final PR review from three independent agent reports.

PR TITLE: {pr_title}
PR DESCRIPTION: {pr_body}

STATIC ANALYSIS REPORT:
{static_analysis}

STYLE REPORT:
{style}

SECURITY REPORT:
{security}

Produce:
1. A concise PR summary (2-3 sentences) — note if the implementation appears to match the stated title/description
2. A suggested commit message (conventional-commits style)
3. Consolidated findings, grouped by severity
4. OVERALL VERDICT: APPROVE, REQUEST_CHANGES, or ESCALATE
   - ESCALATE only if the security report has any high-severity finding
   - REQUEST_CHANGES if there are any warnings/errors from static analysis or style
   - APPROVE only if all three reports are clean
"""

def summarizer_node(state: ReviewState) -> ReviewState:
    result = llm.invoke(summarizer_prompt_v2.format(
        pr_title=state["pr_title"],
        pr_body=state["pr_body"],
        static_analysis=state["static_result"],
        style=state["style_result"],
        security=state["security_result"],
    ))
    return {**state, "final_summary": result.content}

def skip_node(state: ReviewState) -> ReviewState:
    return {**state, "final_summary": f"**Verdict: SKIPPED**\n\n{state['skip_reason']}"}

In [7]:
from langgraph.graph import StateGraph, END

graph_builder = StateGraph(ReviewState)

graph_builder.add_node("gather_context", gather_context_node)
graph_builder.add_node("supervisor", supervisor_node)
graph_builder.add_node("static_analysis", static_analysis_node)
graph_builder.add_node("style", style_node)
graph_builder.add_node("security", security_node)
graph_builder.add_node("summarizer", summarizer_node)
graph_builder.add_node("skip", skip_node)

graph_builder.set_entry_point("gather_context")
graph_builder.add_edge("gather_context", "supervisor")

graph_builder.add_conditional_edges(
    "supervisor",
    route_after_supervisor,
    {"review": "static_analysis", "skip": "skip"},
)

graph_builder.add_edge("static_analysis", "style")
graph_builder.add_edge("style", "security")
graph_builder.add_edge("security", "summarizer")

graph_builder.add_edge("summarizer", END)
graph_builder.add_edge("skip", END)

review_graph = graph_builder.compile()

In [8]:
for branch in ["test/lint-violation", "test/style-violation", "test/security-violation"]:
    pr = next(p for p in repo.get_pulls(state="open") if p.head.ref == branch)
    result = review_graph.invoke({"pr": pr})
    print(f"\n{'='*20} {branch} {'='*20}")
    print(result["final_summary"])


==================== test/lint-violation ====================
**PR Summary**  
The change adds an intentionally unused `uuid` import to `fastapi/applications.py`, which aligns with the PR title’s goal of introducing a lint violation for testing purposes. However, the added import triggers a static‑analysis error and two style warnings (unused import and non‑standard import placement). No security issues were found.

**Suggested Commit Message**  
```
test: introduce intentional unused import to trigger lint violation
```

**Consolidated Findings**

| Severity | Source | File | Line | Finding |
|----------|--------|------|------|---------|
| **Error** | Static Analysis | `fastapi/applications.py` | 4775 | Unused import `uuid` (F401). |
| **Warning** | Style (project‑context) | `fastapi/applications.py` | 4773 | Import placed after a function definition instead of at the top of the module. |
| **Warning** | Style (project‑context) | `fastapi/applications.py` | 4773 | Imported name `_cod